# SIP inflow time-series — monthly trend (Jan 2022 – Dec 2025)

Plotly line chart with an annotation for the **all-time high ₹31,002 Cr** (Dec 2025).

In [1]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

_HERE = Path(__file__).resolve() if '__file__' in globals() else Path.cwd()

def _find_repo_root(start: Path) -> Path:
    cand = start
    for _ in range(10):
        if (cand / 'Data' / 'processed' / 'monthly_sip_inflows_clean.csv').exists():
            return cand
        if cand.name == 'notebooks':
            parent = cand.parent
            if (parent / 'Data' / 'processed' / 'monthly_sip_inflows_clean.csv').exists():
                return parent
        cand = cand.parent
    return start.parent

_REPO_ROOT = _find_repo_root(_HERE)
DATA_PATH = _REPO_ROOT / 'Data' / 'processed' / 'monthly_sip_inflows_clean.csv'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Missing file: {DATA_PATH.resolve()}')

df = pd.read_csv(DATA_PATH)
df['month'] = pd.to_datetime(df['month'], format='%Y-%m', errors='coerce')
df['sip_inflow_crore'] = pd.to_numeric(df['sip_inflow_crore'], errors='coerce')
df = df.dropna(subset=['month', 'sip_inflow_crore']).sort_values('month')

# All-time high from dataset (expected Dec 2025 = 31,002 Cr)
amax = df.loc[df['sip_inflow_crore'].idxmax()].copy()
all_time_high_month = amax['month']
all_time_high_val = float(amax['sip_inflow_crore'])

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df['month'],
        y=df['sip_inflow_crore'],
        mode='lines+markers',
        name='Monthly SIP inflow',
        line=dict(width=3),
        marker=dict(
            size=6,
            color=['#d62728' if t == all_time_high_month else '#4c78a8' for t in df['month']],
        ),
    )
)

fig.add_annotation(
    x=all_time_high_month,
    y=all_time_high_val,
    text=f'All-time high: ₹{all_time_high_val:,.0f} Cr',
    showarrow=True,
    arrowhead=2,
    ax=0,
    ay=-60,
    bgcolor='rgba(255,255,255,0.9)',
    bordercolor='#d62728',
    borderwidth=2,
    font=dict(color='#111'),
)

fig.update_layout(
    title='SIP inflow time-series — monthly trend (Jan 2022 – Dec 2025)',
    template='plotly_white',
    xaxis_title='Month',
    yaxis_title='SIP inflow (Cr)',
    hovermode='x unified',
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
)

fig.show()
